In [10]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sn = pd.read_csv("../_supernova/processed/sn.csv")
bao = pd.read_csv("../_bao/processed/bao.csv")

sn_cov = np.load(
    "../_supernova/processed/sn_covariance.npz"
)["covariance"]
bao_cov = np.load(
    "../_bao/processed/bao_covariance.npz"
)["covariance"]

sn_profile = pd.read_csv("../_supernova/results/sn_profile.csv")
bao_profile = pd.read_csv("../_bao/results/bao_profile.csv")
joint = pd.read_csv("results/joint_summary.csv")

omega_m = sn_profile["Omega_m"].values
assert np.allclose(omega_m, bao_profile["Omega_m"])

om_true = joint.loc[0, "Omega_m"]
offset_true = joint.loc[0, "offset"]
scale_true = joint.loc[0, "A"]

true_index = np.argmin(abs(omega_m - om_true))
assert np.isclose(omega_m[true_index], om_true)

setup_start = time.perf_counter()

In [11]:
def distances(z, om):
    grid = np.linspace(0, max(z), 4001)
    dz = grid[1] - grid[0]
    inv_e = 1 / (om * (1 + grid)**3 + 1 - om)**0.5

    integral = np.zeros(len(grid))
    integral[1:] = np.cumsum((inv_e[:-1] + inv_e[1:]) * dz / 2)

    dm = np.interp(z, grid, integral)
    dh = 1 / (om * (1 + z)**3 + 1 - om)**0.5

    return dm, dh


z_sn = sn["zHD"].values
z_hel = sn["zHEL"].values
z_bao = bao["z"].values
quantity = bao["quantity"].values

sn_shape = np.zeros((len(sn), len(omega_m)))
bao_shape = np.zeros((len(bao), len(omega_m)))

for j, om in enumerate(omega_m):
    dm, dh = distances(z_sn, om)
    sn_shape[:, j] = 5 * np.log10((1 + z_hel) * dm)

    dm, dh = distances(z_bao, om)
    dv = (z_bao * dm**2 * dh)**(1 / 3)

    bao_shape[quantity == "DM_over_rs", j] = dm[quantity == "DM_over_rs"]
    bao_shape[quantity == "DH_over_rs", j] = dh[quantity == "DH_over_rs"]
    bao_shape[quantity == "DV_over_rs", j] = dv[quantity == "DV_over_rs"]

In [12]:
sn_white = np.linalg.solve(
    np.linalg.cholesky(sn_cov),
    np.column_stack((np.ones(len(sn)), sn_shape, sn["m_b_corr"].values)),
)

u = sn_white[:, 0]
sn_templates = sn_white[:, 1:-1]
sn_observed = sn_white[:, -1]
u2 = u @ u

sn_projected = sn_templates - np.outer(u, (u @ sn_templates) / u2)
sn_norm = np.sum(sn_projected**2, axis=0)

bao_white = np.linalg.solve(
    np.linalg.cholesky(bao_cov),
    np.column_stack((bao_shape, bao["value"].values)),
)

bao_templates = bao_white[:, :-1]
bao_observed = bao_white[:, -1]
bao_norm = np.sum(bao_templates**2, axis=0)

sn_mean = sn_templates[:, true_index] + offset_true * u
bao_mean = scale_true * bao_templates[:, true_index]


def fit_profiles(sn_samples, bao_samples):
    sn_centered = sn_samples - np.outer(u, (u @ sn_samples) / u2)

    sn_chi2 = (
        np.sum(sn_centered**2, axis=0)
        + sn_norm.reshape(-1, 1)
        - 2 * (sn_projected.T @ sn_centered)
    )

    bao_chi2 = (
        np.sum(bao_samples**2, axis=0)
        - (bao_templates.T @ bao_samples)**2 / bao_norm.reshape(-1, 1)
    )

    return sn_chi2, bao_chi2

In [13]:
sn_check, bao_check = fit_profiles(
    sn_observed.reshape(-1, 1),
    bao_observed.reshape(-1, 1),
)

assert np.allclose(
    sn_check[:, 0], sn_profile["chi2"].values,
    rtol=0, atol=1e-5,
)
assert np.allclose(
    bao_check[:, 0], bao_profile["chi2"].values,
    rtol=0, atol=1e-5,
)

setup_seconds = time.perf_counter() - setup_start

In [14]:
N_SIM = 2000
SEED = 42

np.random.seed(SEED)
start = time.perf_counter()

sn_mock = sn_mean.reshape(-1, 1) + np.random.normal(
    size=(len(sn), N_SIM)
)
bao_mock = bao_mean.reshape(-1, 1) + np.random.normal(
    size=(len(bao), N_SIM)
)

sn_chi2, bao_chi2 = fit_profiles(sn_mock, bao_mock)
joint_chi2 = sn_chi2 + bao_chi2

simulation = pd.DataFrame({
    "simulation": np.arange(N_SIM),
})

for name, curves in [
    ("sn", sn_chi2),
    ("bao", bao_chi2),
    ("joint", joint_chi2),
]:
    best_values = []
    lower_values = []
    upper_values = []

    for i in range(N_SIM):
        curve = curves[:, i]
        best = np.argmin(curve)
        delta = curve - curve[best]

        assert 0 < best < len(omega_m) - 1
        assert delta[0] > 1 and delta[-1] > 1

        lower = np.interp(
            1, delta[:best + 1][::-1], omega_m[:best + 1][::-1]
        )
        upper = np.interp(
            1, delta[best:], omega_m[best:]
        )

        best_values.append(omega_m[best])
        lower_values.append(lower)
        upper_values.append(upper)

    simulation[name + "_omega_m"] = best_values
    simulation[name + "_lower_68"] = lower_values
    simulation[name + "_upper_68"] = upper_values
    simulation[name + "_covered"] = (
        (simulation[name + "_lower_68"] <= om_true)
        & (simulation[name + "_upper_68"] >= om_true)
    )

simulation["delta_chi2"] = (
    np.min(joint_chi2, axis=0)
    - np.min(sn_chi2, axis=0)
    - np.min(bao_chi2, axis=0)
)

elapsed = time.perf_counter() - start

simulation.to_csv("results/monte_carlo_50.csv", index=False)

In [15]:
timing = pd.DataFrame({
    "simulations": [N_SIM],
    "setup_seconds": [setup_seconds],
    "simulation_seconds": [elapsed],
    "estimated_2000_minutes": [elapsed * 2000 / N_SIM / 60],
})

timing.to_csv("results/monte_carlo_50_timing.csv", index=False)

timing

,simulations,setup_seconds,simulation_seconds,estimated_2000_minutes
0,2000,1.132538,0.097143,0.001619


In [16]:
rows = []

for name in ["sn", "bao", "joint"]:
    estimates = simulation[name + "_omega_m"]

    rows.append({
        "data": name,
        "Omega_m_input": om_true,
        "Omega_m_mean": np.mean(estimates),
        "mean_minus_input": np.mean(estimates) - om_true,
        "coverage_68": np.mean(simulation[name + "_covered"]),
        "mean_interval_width": np.mean(
            simulation[name + "_upper_68"]
            - simulation[name + "_lower_68"]
        ),
    })

recovery = pd.DataFrame(rows)
recovery.to_csv("results/monte_carlo_50_recovery.csv", index=False)

recovery

,data,Omega_m_input,Omega_m_mean,mean_minus_input,coverage_68,mean_interval_width
0,sn,0.304,0.304208,0.000209,0.6830,0.034857
1,bao,0.304,0.303772,-0.000228,0.6910,0.017556
2,joint,0.304,0.303798,-0.000202,0.6875,0.015674


In [17]:
observed_delta = (
    min(sn_profile["chi2"].values + bao_profile["chi2"].values)
    - min(sn_profile["chi2"])
    - min(bao_profile["chi2"])
)

exceed_count = sum(simulation["delta_chi2"] >= observed_delta)

consistency = pd.DataFrame({
    "observed_delta_chi2": [observed_delta],
    "exceed_count": [exceed_count],
    "simulations": [N_SIM],
    "exceed_fraction": [exceed_count / N_SIM],
})

consistency.to_csv("results/monte_carlo_50_consistency.csv", index=False)

consistency

,observed_delta_chi2,exceed_count,simulations,exceed_fraction
0,2.946395,167,2000,0.0835
